# Código de Repetición: Detectando Errores sin Destruir Superposición

Vamos a construir un código de bit-flip y extraer medidas de paridad en Qiskit.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

## 1. El Circuito: 3 Datos lógicos y 2 de Síndrome (Ancillas)
Codificamos el estado protegido, inducimos un error malévolo manualmente y usamos los qubits ancilares para detectar quién se ha equivocado mediante CNOTs reversantes diferenciales.

In [ ]:
qr_data = QuantumRegister(3, 'data')
qr_synd = QuantumRegister(2, 'ancilla_sindrome')
cr_med = ClassicalRegister(2, 'medicion_sindrome')

qc = QuantumCircuit(qr_data, qr_synd, cr_med)

# Preparamos un estado arbitrario (rotacion +)
qc.h(qr_data[0])

# Codificamos lógicamente este qubit de la posición 0 hacia los qubits de datos 1 y 2
qc.cx(qr_data[0], qr_data[1])
qc.cx(qr_data[0], qr_data[2])

qc.barrier()
# ¡PUM! SIMULAMOS UN BIT-FLIP ERROR (Rayos cósmicos)
# Cambiamos bruscamente el segundo qubit de los datos
qc.x(qr_data[1])
qc.barrier()

# MEDICIÓN SÍNDROME: 
# Ancilares hacen XOR comparativo. Ancilla 0 compara Data 0 y 1. Ancilla 1 compara Data 1 y 2.
qc.cx(qr_data[0], qr_synd[0])
qc.cx(qr_data[1], qr_synd[0])

qc.cx(qr_data[1], qr_synd[1])
qc.cx(qr_data[2], qr_synd[1])

# Solo leemos los de Síndrome, sin alterar el estado original en la base de datos!
qc.measure(qr_synd, cr_med)

display(qc.draw('mpl'))

## 2. Ejecutar y Descifrar el Síndrome
Si obtenemos paridad $1$ de la ancila 0, deducimos que los pares (0,1) son dispares. Si obtenemos un $1$ en la ancila 1, determinamos que los pares (1,2) son dispares. Por trigonometría booleana, el que se equivocó debió ser ¡el del centro (el registro 1)!

In [ ]:
sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)
res = job.result()
counts = res[0].data.medicion_sindrome.get_counts()

print(f"Cuentas del síndrome recogido: {counts}")
print("Efectivamente la inmensa mayoría arrojó '11'. \nSignifica que Ambos comparadores de paridad detectaron una falla en su vínculo compartido (el qubit en la posición 1 que alteramos intencionadamente). Ahora se aplicaría lógicamente un qc.x(1) correctivo y nuestro qubit mágico de superposición quedaría a salvo de este ruido sin haber medido jamás su valor crudo (superposición intacta).")